# Data Quality Report

Este notebook consolida as verificações automatizadas de qualidade de dados do projeto, incluindo nulos, duplicatas, integridade referencial, outliers e inconsistências temporais.

A maior parte das checagens é implementada como testes do dbt. Ao final, há algumas checagens em tabelas que ficaram fora do modelo dimensional (`geolocation`, `product_category_name_translation`), mas que ainda fazem parte do dataset bruto.

## Como reproduzir

Pré-requisito: o dataset já deve ter sido ingerido no DuckDB (`python src/ingest.py`, a partir da raiz do projeto).

Este notebook roda `dbt build` diretamente e lê o resultado estruturado que o dbt gera automaticamente após a execução.

In [1]:
import json
import subprocess
import sys
from pathlib import Path
import duckdb
import pandas as pd

In [2]:
PROJECT_ROOT = Path("..").resolve()
DBT_PROJECT_DIR = PROJECT_ROOT / "dbt_project"

dbt_executable = Path(sys.executable).parent / "dbt.exe"

run_results_path = DBT_PROJECT_DIR / "target" / "run_results.json"
run_results_path.unlink(missing_ok=True)

resultado = subprocess.run(
    [
        str(dbt_executable), "build",
        "--project-dir", str(DBT_PROJECT_DIR),
        "--profiles-dir", str(DBT_PROJECT_DIR),
    ],
    capture_output=True,
    text=True,
)

print(resultado.stdout[-2500:])

if resultado.returncode != 0:
    raise RuntimeError(
        f"dbt build falhou (returncode={resultado.returncode}). "
        "O notebook não deve continuar com resultado desatualizado."
    )

s]
18:49:52  52 of 54 START test not_null_fct_reviews_order_id .............................. [RUN]
18:49:52  48 of 54 PASS relationships_fct_order_items_product_id__product_id__ref_dim_products_  [PASS in 0.10s]
18:49:52  53 of 54 START test relationships_fct_reviews_order_id__order_id__ref_dim_orders_  [RUN]
18:49:52  51 of 54 PASS after_column_fct_reviews_review_answer_timestamp__review_creation_date  [PASS in 0.05s]
18:49:52  50 of 54 PASS accepted_values_fct_reviews_review_score__1__2__3__4__5 .......... [PASS in 0.06s]
18:49:52  54 of 54 START test unique_combination_of_columns_fct_reviews_review_id__order_id  [RUN]
18:49:52  52 of 54 PASS not_null_fct_reviews_order_id .................................... [PASS in 0.05s]
18:49:52  53 of 54 PASS relationships_fct_reviews_order_id__order_id__ref_dim_orders_ .... [PASS in 0.06s]
18:49:52  54 of 54 PASS unique_combination_of_columns_fct_reviews_review_id__order_id .... [PASS in 0.05s]
18:49:52  
18:49:52  Finished running 7 table mod

## Resumo dos testes automatizados (dbt)

In [3]:
with open(DBT_PROJECT_DIR / "target" / "run_results.json", encoding="utf-8") as f:
    run_results = json.load(f)


def categorizar(nome_teste: str) -> str:
    """Classifica o teste do dbt em uma categoria de qualidade de dados."""
    if nome_teste.startswith("not_null"):
        return "Nulos"
    if nome_teste.startswith("unique"):
        return "Duplicatas"
    if nome_teste.startswith("relationships"):
        return "Integridade Referencial"
    if nome_teste.startswith("accepted_values") or nome_teste.startswith("non_negative"):
        return "Outliers"
    if nome_teste.startswith("after_column"):
        return "Inconsistência Temporal"
    if nome_teste.startswith("assert_payments_match"):
        return "Consistência entre Tabelas"
    return "Outro"


linhas = []
for r in run_results["results"]:
    if not r["unique_id"].startswith("test."):
        continue
    nome_teste = r["unique_id"].split(".")[2]
    linhas.append({
        "teste": nome_teste,
        "categoria": categorizar(nome_teste),
        "status": r["status"],
        "mensagem": r.get("message") or "",
    })

testes_df = pd.DataFrame(linhas)
testes_df

,teste,categoria,status,mensagem
0,not_null_dim_customers_customer_id,Nulos,pass,
1,accepted_values_dim_customers_customer_state__...,Outliers,pass,
2,not_null_dim_customers_customer_unique_id,Nulos,pass,
3,unique_dim_customers_customer_id,Duplicatas,pass,
4,accepted_values_dim_orders_order_status__deliv...,Outliers,pass,
5,after_column_dim_orders_order_approved_at__ord...,Inconsistência Temporal,pass,
6,after_column_dim_orders_order_delivered_carrie...,Inconsistência Temporal,warn,"Got 1359 results, configured to warn if != 0"
7,after_column_dim_orders_order_delivered_custom...,Inconsistência Temporal,warn,"Got 23 results, configured to warn if != 0"
8,not_null_dim_orders_customer_id,Nulos,pass,
9,after_column_dim_orders_order_delivered_custom...,Inconsistência Temporal,pass,


In [4]:
resumo = testes_df.groupby(["categoria", "status"]).size().unstack(fill_value=0)
resumo

status,pass,warn
categoria,,
Consistência entre Tabelas,0,1
Duplicatas,7,0
Inconsistência Temporal,3,2
Integridade Referencial,6,0
Nulos,13,0
Outliers,8,0


### Avisos conhecidos

Três testes estão configurados como `warn`, pois representam achados conhecidos e documentados nos dados:

- Aprovação fora de ordem (`order_delivered_carrier_date` antes de `order_approved_at`)
- Entrega antes do despacho (`order_delivered_customer_date` antes de `order_delivered_carrier_date`)
- Divergência de pagamento vs. itens: o total pago diverge do total de item+frete em uma pequena fração dos pedidos (cerca de 0,25% dos casos), porém, isso pode ser ocasionado por algum desconto ou taxa extra não contabilizados. Dessa forma, optou-se por manter o alerta como warn, e não como error.

Por fim, vale ressaltar que o teste que compara a data de entrega final com a data de compra inicial continua configurado como `error`, já que a divergência entre essas datas pode causas problemas maiores e estruturais.

In [5]:
testes_df[testes_df["status"] == "warn"][["teste", "categoria", "mensagem"]]

,teste,categoria,mensagem
6,after_column_dim_orders_order_delivered_carrie...,Inconsistência Temporal,"Got 1359 results, configured to warn if != 0"
7,after_column_dim_orders_order_delivered_custom...,Inconsistência Temporal,"Got 23 results, configured to warn if != 0"
19,assert_payments_match_order_items_total,Consistência entre Tabelas,"Got 249 results, configured to warn if != 0"


## Achados complementares (tabelas fora do modelo dimensional)

`geolocation` e `product_category_name_translation` não fazem parte do modelo (motivos documentados no README), então não apresentam testes no dbt. Abaixo, seguem algumas informações sobre a qualidade das tabelas.

In [6]:
geo_path = PROJECT_ROOT / "data" / "raw" / "olist_geolocation_dataset.csv"
geo = duckdb.sql(f"SELECT * FROM read_csv_auto(\'{geo_path.as_posix()}\')").df()

duplicadas = geo.duplicated().sum()
lat_fora = ((geo["geolocation_lat"] < -34) | (geo["geolocation_lat"] > 6)).sum()
lng_fora = ((geo["geolocation_lng"] < -75) | (geo["geolocation_lng"] > -32)).sum()

print(f"geolocation - linhas totalmente duplicadas: {duplicadas}")
print(f"geolocation - latitudes fora do intervalo esperado do Brasil: {lat_fora}")
print(f"geolocation - longitudes fora do intervalo esperado do Brasil: {lng_fora}")

geolocation - linhas totalmente duplicadas: 261831
geolocation - latitudes fora do intervalo esperado do Brasil: 31
geolocation - longitudes fora do intervalo esperado do Brasil: 26


In [7]:
translation_path = PROJECT_ROOT / "data" / "raw" / "product_category_name_translation.csv"
products_path = PROJECT_ROOT / "data" / "raw" / "olist_products_dataset.csv"

translation = duckdb.sql(f"SELECT * FROM read_csv_auto(\'{translation_path.as_posix()}\')").df()
products = duckdb.sql(f"SELECT * FROM read_csv_auto(\'{products_path.as_posix()}\')").df()

duplicadas_translation = translation.duplicated().sum()
categorias_sem_traducao = sorted(
    set(products["product_category_name"].dropna())
    - set(translation["product_category_name"])
)

print(f"product_category_name_translation - linhas duplicadas: {duplicadas_translation}")
print(f"Categorias de products sem tradução correspondente: {categorias_sem_traducao}")

product_category_name_translation - linhas duplicadas: 0
Categorias de products sem tradução correspondente: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


## Conclusão

In [8]:
total = len(testes_df)
por_status = testes_df["status"].value_counts()

print(f"Total de testes automatizados no dbt: {total}")
for status, contagem in por_status.items():
    print(f"  {status}: {contagem}")
print()
print(f"Categorias cobertas: {sorted(testes_df['categoria'].unique())}")

Total de testes automatizados no dbt: 40
  pass: 37
  warn: 3

Categorias cobertas: ['Consistência entre Tabelas', 'Duplicatas', 'Inconsistência Temporal', 'Integridade Referencial', 'Nulos', 'Outliers']


Foram feitos 40 testes, cobrindo seis categorias: consistência entre tabelas, duplicatas, inconsistência temporal, integridade referencial, nulos e outliers.

Decisões de modelagem e detalhamentos estão em `README.md` e `docs/erd.md`.